### Modell anzeigen

In [1]:
from core.config import LLM_MODEL, OLLAMA_URL
print("LLM_MODEL =", LLM_MODEL)
print("OLLAMA_URL =", OLLAMA_URL)

LLM_MODEL = gpt-oss:120b-cloud
OLLAMA_URL = http://localhost:11434


### Aus ChromaDB `document_id` ausgeben:

In [ ]:
from langchain_chroma import Chroma
from chromadb.config import Settings

from utils.ollama_embed import OllamaEmbeddings
from core.config import PATH_DB, COLLECTION, EMBED_MODEL, OLLAMA_URL, PATH_PROCESSED

# ------------------------------------------------------------------
# Create embedding model
# ------------------------------------------------------------------
embeddings = OllamaEmbeddings(
    model=EMBED_MODEL,
    base_url=OLLAMA_URL
)

# ------------------------------------------------------------------
# Connect to vector database
# ------------------------------------------------------------------
vector_db = Chroma(
    collection_name=COLLECTION,
    persist_directory=PATH_DB,
    embedding_function=embeddings,
    client_settings=Settings(anonymized_telemetry=False),
)

# --- Nur Dokument-IDs anzeigen ---
res = vector_db.get(include=["metadatas"])
metas = res.get("metadatas", []) or []

doc_ids = [m.get("source") for m in metas if m and m.get("source")]
unique_ids = list(dict.fromkeys(doc_ids))  # Reihenfolge erhalten

print(f"{len(list(PATH_PROCESSED.glob('*.md')))} Markdown-Dateien in data/processed")
print(f"{len(unique_ids)} insgesamt in DB-Collection:\n")

i=1
for doc in unique_ids:
    print(f"{i}. {doc}")
    i = i + 1

### Test Chunking

In [2]:
# utils/chunking.py
from langchain_text_splitters import MarkdownHeaderTextSplitter
from utils.logger import logger

def chunk_documents(docs):
    logger.info("Chunking beginnt ...")

    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[("#", "H1"), ("##", "H2"), ("###", "H3")],
        strip_headers=False
    )

    chunks = [] # An empty list is created in which all generated chunks are collected.
    for doc in docs:
        for chunk in splitter.split_text(doc.page_content):
            chunk.metadata["source"] = doc.metadata.get("source") # Add  metadata `source` for example: `dokument_1.md`
            chunks.append(chunk) # Save the new chunk

    logger.info("Chunking abgeschlossen.")
    return chunks


In [3]:
markdown_document = """
    # Das ist Header 1 üüü
    text_Header1

    ## Das ist Header 2
    text_Header2

    ### Das ist Header 3
    text_Header3
"""

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on, strip_headers=False)
md_header_splits = markdown_splitter.split_text(markdown_document)
md_header_splits

[Document(metadata={'Header 1': 'Das ist Header 1 üüü'}, page_content='# Das ist Header 1 üüü\ntext_Header1'),
 Document(metadata={'Header 1': 'Das ist Header 1 üüü', 'Header 2': 'Das ist Header 2'}, page_content='## Das ist Header 2\ntext_Header2'),
 Document(metadata={'Header 1': 'Das ist Header 1 üüü', 'Header 2': 'Das ist Header 2', 'Header 3': 'Das ist Header 3'}, page_content='### Das ist Header 3\ntext_Header3')]